# Business Challenge Report

Data-driven executive summary built on the `tpch_dev.gold` layer, leveraging outputs from the
Bronze→Silver→Gold Lakeflow pipeline, 20-query SQL analysis, and DQ reconciliation report.

* **3 Key Findings** — each backed by a specific query result and metric value
* **2 Opportunities** for growth or operational improvement, with data evidence
* **2 Risks** requiring further investigation, with data evidence
* **Recommended management action** for every item

In [0]:
%sql
-- Finding 1: Revenue distribution across customer segments and regions
SELECT 'BY_SEGMENT' AS dimension, market_segment AS category,
  ROUND(net_revenue, 2) AS net_revenue, total_orders
FROM (
  SELECT dc.customer_mktsegment AS market_segment,
    SUM(fs.net_revenue) AS net_revenue,
    COUNT(DISTINCT fs.order_key) AS total_orders
  FROM tpch_dev.gold.fact_sales fs
  JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
  GROUP BY dc.customer_mktsegment
)
UNION ALL
SELECT 'BY_REGION', customer_region,
  ROUND(net_revenue, 2), total_orders
FROM (
  SELECT fs.customer_region_name AS customer_region,
    SUM(fs.net_revenue) AS net_revenue,
    COUNT(DISTINCT fs.order_key) AS total_orders
  FROM tpch_dev.gold.fact_sales fs
  GROUP BY fs.customer_region_name
)
ORDER BY dimension, net_revenue DESC

### Finding 1 — Revenue is highly diversified; no single segment or region dominates

**Data Evidence (from query above):**

| Dimension | Highest | Lowest | Spread |
|-----------|--------|--------|--------|
| **Segment** | FURNITURE: $227.77B | HOUSEHOLD: $225.30B | $2.47B (1.1%) |
| **Region** | EUROPE: $227.71B | AFRICA: $225.35B | $2.37B (1.0%) |

All five market segments fall within a **1.1% band** of each other, and all five regions within a **1.0% band**. No single segment or region accounts for more than 20.1% of total revenue ($1.13T). This indicates an extremely well-diversified customer base.

**Recommended Management Action:**
Maintain the diversified portfolio by avoiding over-investment in any single segment. Allocate marketing budgets proportionally (≈20% per segment, ≈20% per region). Use the [Business Query notebook](#notebook-4173249402066206) Q04 and Q12 results for ongoing tracking.

In [0]:
%sql
-- Finding 2: Revenue and volume split by return flag
SELECT
  fs.return_flag,
  ROUND(SUM(fs.net_revenue), 2) AS net_revenue,
  ROUND(SUM(fs.net_revenue) * 100 / SUM(SUM(fs.net_revenue)) OVER (), 2) AS pct_revenue,
  SUM(fs.quantity) AS total_quantity,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
GROUP BY fs.return_flag
ORDER BY net_revenue DESC

### Finding 2 — Nearly half of all revenue involves returned items

**Data Evidence (from query above):**

| Return Flag | Net Revenue | % of Total | Line Items |
|-------------|-------------|-----------|------------|
| **N** (not returned) | $573.81B | 50.63% | 15,189,553 |
| **R** (returned) | $279.91B | 24.70% | 7,406,353 |
| **A** (accepted return) | $279.71B | 24.68% | 7,403,889 |

Combined, **R + A = 49.38%** of all revenue ($559.63B) involves a return event. The returned and accepted-return categories are almost perfectly balanced (24.70% vs 24.68%), suggesting a systematic return processing workflow rather than ad-hoc exceptions.

**Recommended Management Action:**
Initiate a returns-quality task force to investigate root causes across product lines and suppliers. Target reducing R+A from 49.4% to below 40% within two quarters. Cross-reference returned line items with supplier data (Q10/Q11 in [Business Query](#notebook-4173249402066206)) to identify suppliers with disproportionate return rates.

In [0]:
%sql
-- Finding 3: Top 10 customer and supplier concentration
WITH top_customers AS (
  SELECT SUM(net_revenue) AS combined_revenue
  FROM (
    SELECT dc.customer_key, SUM(fs.net_revenue) AS net_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
    GROUP BY dc.customer_key
    ORDER BY net_revenue DESC
    LIMIT 10
  )
),
top_suppliers AS (
  SELECT SUM(net_revenue) AS combined_revenue
  FROM (
    SELECT ds.supplier_key, SUM(fs.net_revenue) AS net_revenue
    FROM tpch_dev.gold.fact_sales fs
    JOIN tpch_dev.gold.dim_supplier ds ON fs.supplier_key = ds.supplier_key
    GROUP BY ds.supplier_key
    ORDER BY net_revenue DESC
    LIMIT 10
  )
),
total AS (
  SELECT SUM(net_revenue) AS grand_total FROM tpch_dev.gold.fact_sales
)
SELECT 'Top 10 Customers' AS entity,
  ROUND(tc.combined_revenue, 2) AS combined_revenue,
  ROUND(tc.combined_revenue / t.grand_total * 100, 4) AS pct_of_total,
  (SELECT COUNT(DISTINCT customer_key) FROM tpch_dev.gold.fact_sales) AS total_active_entities
FROM top_customers tc, total t
UNION ALL
SELECT 'Top 10 Suppliers',
  ROUND(ts.combined_revenue, 2),
  ROUND(ts.combined_revenue / t.grand_total * 100, 4),
  (SELECT COUNT(DISTINCT supplier_key) FROM tpch_dev.gold.fact_sales)
FROM top_suppliers ts, total t

### Finding 3 — Zero concentration risk: top 10 customers and suppliers each represent <0.1% of revenue

**Data Evidence (from query above):**

| Entity | Combined Revenue (Top 10) | % of Total ($1.13T) | Total Active Entities |
|--------|--------------------------|---------------------|----------------------|
| **Top 10 Customers** | ~$69.9M | 0.0062% | 499,989 |
| **Top 10 Suppliers** | ~$300.5M | 0.0265% | 50,000 |

With nearly 500K active customers and 50K suppliers, the top 10 of each represent a negligible share. The largest individual customer generates only ~$7.1M — just 0.0006% of total revenue. There is no single-point-of-failure dependency.

**Recommended Management Action:**
The lack of anchor accounts means customer acquisition and retention at scale is critical. Invest in self-service and automated engagement channels rather than high-touch account management. Monitor the top-customer share quarterly using Q06 in [Business Query](#notebook-4173249402066206) — if it rises above 5%, reassess concentration risk.

In [0]:
%sql
-- Opportunity 1: Top 10 vs Bottom 10 product revenue gap
WITH product_revenue AS (
  SELECT dp.product_key, dp.product_name, dp.product_type,
    SUM(fs.net_revenue) AS net_revenue,
    SUM(fs.quantity) AS total_quantity
  FROM tpch_dev.gold.fact_sales fs
  JOIN tpch_dev.gold.dim_product dp ON fs.product_key = dp.product_key
  GROUP BY dp.product_key, dp.product_name, dp.product_type
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (ORDER BY net_revenue DESC) AS rank_top,
    ROW_NUMBER() OVER (ORDER BY net_revenue ASC)  AS rank_bottom
  FROM product_revenue
)
SELECT 'TOP 10' AS tier,
  ROUND(SUM(net_revenue), 2) AS combined_revenue,
  ROUND(AVG(net_revenue), 2)  AS avg_per_product,
  ROUND(AVG(total_quantity), 0) AS avg_quantity
FROM ranked WHERE rank_top <= 10
UNION ALL
SELECT 'BOTTOM 10',
  ROUND(SUM(net_revenue), 2),
  ROUND(AVG(net_revenue), 2),
  ROUND(AVG(total_quantity), 0)
FROM ranked WHERE rank_bottom <= 10
UNION ALL
SELECT 'ALL PRODUCTS',
  ROUND(SUM(net_revenue), 2),
  ROUND(AVG(net_revenue), 2),
  ROUND(AVG(total_quantity), 0)
FROM product_revenue

### Opportunity 1 — Rationalize the long tail: bottom 10 products earn 88% less than top 10

**Data Evidence (from query above):**

| Tier | Combined Revenue | Avg per Product | Avg Quantity |
|------|-----------------|-----------------|-------------|
| **Top 10** | ~$28.9M | ~$2.89M | ~1,484 units |
| **Bottom 10** | ~$2.4M | ~$0.24M | ~219 units |
| **All Products (1M)** | $1.13T | ~$1.13K | — |

The bottom 10 products average only **$240K** each — roughly **12× less** than the top 10 at $2.89M. Across 1 million products, the long tail is vast. Retiring or repositioning the lowest performers could free up inventory, catalog space, and supplier management bandwidth.

**Recommended Management Action:**
Implement a quarterly product performance review: any product below the 5th percentile of revenue for two consecutive quarters is flagged for discontinuation or repositioning. Redirect freed supplier capacity toward top-performing product types (see Q09 in [Business Query](#notebook-4173249402066206)).

In [0]:
%sql
-- Opportunity 2: Shipping mode efficiency comparison
SELECT
  fs.ship_mode,
  ROUND(AVG(DATEDIFF(fs.receipt_date, fs.ship_date)), 1) AS avg_ship_to_receipt_days,
  COUNT(*) AS line_count,
  ROUND(SUM(fs.net_revenue), 2) AS revenue_via_mode
FROM tpch_dev.gold.fact_sales fs
WHERE fs.ship_date IS NOT NULL AND fs.receipt_date IS NOT NULL
GROUP BY fs.ship_mode
ORDER BY revenue_via_mode DESC

### Opportunity 2 — All shipping modes deliver in 15.5 days; mode choice has no speed advantage

**Data Evidence (from query above):**

| Ship Mode | Avg Receipt Days | Line Count | Revenue via Mode |
|-----------|-----------------|------------|------------------|
| TRUCK | 15.5 | 4,288,377 | ~$162.6B |
| FOB | 15.5 | 4,287,168 | ~$162.6B |
| MAIL | 15.5 | 4,282,860 | ~$162.6B |
| RAIL | 15.5 | 4,284,870 | ~$162.6B |
| REG AIR | 15.5 | 4,285,596 | ~$162.6B |
| SHIP | 15.5 | 4,285,381 | ~$162.6B |
| AIR | 15.5 | 4,285,543 | ~$162.6B |

Every shipping mode has an **identical 15.5-day** average ship-to-receipt time, and each carries roughly the same volume (~4.28M line items). This means premium modes (AIR, REG AIR) deliver no speed benefit over cheaper modes (TRUCK, RAIL, MAIL).

**Recommended Management Action:**
Shift volume from premium modes (AIR, REG AIR) to lower-cost modes (TRUCK, RAIL, MAIL) for non-time-sensitive orders. Since delivery times are identical, this would reduce shipping costs without impacting customer experience. Target a 20% modal shift in Q1 and monitor for any change in receipt times.

In [0]:
%sql
-- Risk 1: Annual revenue with year-over-year percentage change
SELECT
  YEAR(order_date) AS sales_year,
  ROUND(SUM(net_revenue), 2) AS annual_revenue,
  LAG(ROUND(SUM(net_revenue), 2)) OVER (ORDER BY YEAR(order_date)) AS prev_year_revenue,
  ROUND(
    (SUM(net_revenue) - LAG(SUM(net_revenue)) OVER (ORDER BY YEAR(order_date)))
    / LAG(SUM(net_revenue)) OVER (ORDER BY YEAR(order_date)) * 100, 2
  ) AS yoy_pct_change
FROM tpch_dev.gold.fact_sales
GROUP BY YEAR(order_date)
ORDER BY sales_year

### Risk 1 — 1998 revenue shows a 41.3% YoY decline; requires urgent investigation

**Data Evidence (from query above):**

| Year | Annual Revenue | YoY % Change |
|------|---------------|-------------|
| 1992 | $172.18B | — |
| 1993 | $172.23B | +0.03% |
| 1994 | $172.08B | -0.09% |
| 1995 | $171.63B | -0.26% |
| 1996 | $172.61B | +0.57% |
| 1997 | $171.86B | -0.43% |
| **1998** | **$100.85B** | **-41.32%** |

Revenue was remarkably stable from 1992–1997 (±0.6%), then **dropped 41.3%** in 1998. This could be partial-year data (the dataset may end mid-1998) or a genuine business contraction. The DQ reconciliation report confirms row counts reconcile between Silver and Gold, so this is not a data pipeline issue.

**Recommended Management Action:**
Verify whether 1998 data covers a full 12 months or a partial period by checking the max `order_date` in `fact_sales`. If partial, annualize the run-rate. If full, investigate root causes — customer attrition, supply disruption, or market shift — and compare with external market data for 1998.

In [0]:
%sql
-- Risk 2: Discount uniformity across market segments
SELECT
  dc.customer_mktsegment AS market_segment,
  ROUND(AVG(fs.discount) * 100, 2) AS avg_discount_pct,
  ROUND(AVG(fs.discount_amount), 2) AS avg_discount_amount,
  ROUND(SUM(fs.net_revenue), 2) AS segment_revenue,
  ROUND(SUM(fs.discount_amount), 2) AS total_discount_given,
  COUNT(*) AS line_count
FROM tpch_dev.gold.fact_sales fs
JOIN tpch_dev.gold.dim_customer dc ON fs.customer_key = dc.customer_key
GROUP BY dc.customer_mktsegment
ORDER BY avg_discount_pct DESC

### Risk 2 — Flat 5% discount across all segments regardless of volume; margin erosion risk

**Data Evidence (from query above):**

| Segment | Avg Discount % | Avg Discount $ | Total Discount Given |
|---------|---------------|---------------|---------------------|
| HOUSEHOLD | 5.00% | $1,911.48 | ~$11.4B |
| AUTOMOBILE | 5.00% | $1,910.82 | ~$11.4B |
| BUILDING | 5.00% | $1,912.48 | ~$11.5B |
| MACHINERY | 5.00% | $1,911.91 | ~$11.5B |
| FURNITURE | 5.00% | $1,912.68 | ~$11.5B |

Every segment receives an **identical 5.00% average discount**, with per-line discount amounts varying by less than $2. Total discounts given amount to **~$57.3B** across all segments. This uniform discounting suggests a one-size-fits-all policy that does not differentiate by customer value, order volume, or segment profitability.

**Recommended Management Action:**
Introduce a tiered discount framework: high-volume segments or customers receive volume-based rebates instead of flat discounts, while low-volume segments are moved to list-price-plus models. Target reducing the total discount pool by 10% ($5.7B) over two quarters by eliminating discounts on low-margin product types. Track impact via Q15 in [Business Query](#notebook-4173249402066206).

## Executive Summary

| # | Category | Title | Key Metric | Recommended Action |
|---|----------|-------|-----------|--------------------|
| F1 | **Finding** | Revenue highly diversified | Segments within 1.1%; regions within 1.0% | Maintain proportional budget allocation (≈20% per segment/region) |
| F2 | **Finding** | 49.4% of revenue involves returns | R+A = $559.63B (49.38%) | Launch returns-quality task force; target <40% R+A in two quarters |
| F3 | **Finding** | Zero customer/supplier concentration | Top 10 customers = 0.006% of revenue | Invest in self-service channels; monitor concentration quarterly |
| O1 | **Opportunity** | Long-tail product rationalization | Bottom 10 avg $240K vs top 10 avg $2.89M (12× gap) | Quarterly performance review; retire/reposition below 5th percentile |
| O2 | **Opportunity** | Shipping mode optimization | All modes = 15.5 days; no speed advantage | Shift 20% volume from AIR/REG AIR to TRUCK/RAIL/MAIL in Q1 |
| R1 | **Risk** | 1998 revenue dropped 41.3% YoY | $100.85B vs $171.86B prior year | Verify partial vs full year; investigate root causes if full |
| R2 | **Risk** | Flat 5% discount erodes margins | $57.3B total discounts, uniform across segments | Introduce tiered discount framework; target 10% discount-pool reduction |

---
*This report leverages the `tpch_dev.gold` layer built by the Bronze→Silver→Gold Lakeflow pipeline, validated by the [DQ & Reconciliation report](#notebook-4173249402066205), and informed by the [20-query SQL analysis](#notebook-4173249402066206).*